# Colab: Generate Emotional Story Datasets

This notebook runs the emotional-story generator on Google Colab.

It uses `scripts/generate_datasets.py`, which is a thin wrapper around the current generator implementation at `scripts/my_dataset/generate_dataset.py`.

Use `hf` on Colab. If you do not have a very large GPU, start with `Qwen/Qwen2.5-7B-Instruct` or `Qwen/Qwen2.5-14B-Instruct` instead of `32B`.

## 1. Repo and Runtime

Run the next three cells first. They clone the repo into `/content`, install the Colab-side Python packages, and show the attached GPU.

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/daspushpita/emotion-mechanisms-llm.git"
REPO_DIR = Path("/content/emotion-mechanisms-llm")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only

%cd /content/emotion-mechanisms-llm

In [ ]:
!pip install -q -U transformers accelerate huggingface_hub sentencepiece

In [ ]:
!nvidia-smi

## 2. Configure the Run

Edit the values in the next cell before you launch generation.

- `HF_MODEL_ID`: pick the Hugging Face model to use on Colab.
- `LIMIT_TOPICS`: set this to a small number for a smoke test.
- `SAMPLES_PER_TOPIC_EMOTION`, `STORIES_PER_BATCH`, and `MAX_NEW_TOKENS`: optional overrides for a smaller or faster run.
- `MOUNT_DRIVE`: mount Google Drive so you can copy the final JSONL out of ephemeral Colab storage.

In [ ]:
HF_MODEL_ID = "Qwen/Qwen2.5-32B-Instruct"
TOPICS_FILE = "datasets/raw/topics.txt"

LIMIT_TOPICS = None
SAMPLES_PER_TOPIC_EMOTION = None
STORIES_PER_BATCH = None
MAX_NEW_TOKENS = None

MOUNT_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/emotion-mechanisms-llm/datasets/processed"

In [ ]:
from pathlib import Path

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUTPUT_DIR = Path(DRIVE_OUTPUT_DIR)
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Drive output dir: {DRIVE_OUTPUT_DIR}")
else:
    DRIVE_OUTPUT_DIR = None
    print("Skipping Google Drive mount.")

## 3. Optional Hugging Face Login

Run the next cell only if the model download needs authentication or you want authenticated rate limits.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 4. Run Dataset Generation

The next two cells prepare environment overrides for the wrapper script and then run generation.

In [ ]:
import os

topics_path = REPO_DIR / TOPICS_FILE

if LIMIT_TOPICS is not None:
    subset_path = REPO_DIR / "datasets/raw/topics_colab_subset.txt"
    topics = [line.strip() for line in topics_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    subset_path.write_text("\n".join(topics[:LIMIT_TOPICS]) + "\n", encoding="utf-8")
    topics_path = subset_path
    print(f"Using a subset of {LIMIT_TOPICS} topics: {topics_path}")
else:
    print(f"Using full topics file: {topics_path}")

env = os.environ.copy()
env["EMOTION_MODEL"] = "hf"
env["HF_MODEL_ID"] = HF_MODEL_ID
env["DATASET_TOPICS_PATH"] = str(topics_path)
env["DATASET_PROCESSED_DIR"] = str(REPO_DIR / "datasets/processed")

if SAMPLES_PER_TOPIC_EMOTION is not None:
    env["DATASET_SAMPLES_PER_TOPIC_EMOTION"] = str(SAMPLES_PER_TOPIC_EMOTION)
if STORIES_PER_BATCH is not None:
    env["DATASET_STORIES_PER_BATCH"] = str(STORIES_PER_BATCH)
if MAX_NEW_TOKENS is not None:
    env["DATASET_MAX_NEW_TOKENS"] = str(MAX_NEW_TOKENS)

for key in [
    "EMOTION_MODEL",
    "HF_MODEL_ID",
    "DATASET_TOPICS_PATH",
    "DATASET_PROCESSED_DIR",
    "DATASET_SAMPLES_PER_TOPIC_EMOTION",
    "DATASET_STORIES_PER_BATCH",
    "DATASET_MAX_NEW_TOKENS",
]:
    if key in env:
        print(f"{key}={env[key]}")

In [ ]:
import subprocess

subprocess.run(
    ["python3", "scripts/generate_datasets.py"],
    cwd=REPO_DIR,
    env=env,
    check=True,
)

## 5. Inspect and Persist the Output

This shows the generated file and optionally copies it into Google Drive.

In [ ]:
import shutil

output_path = REPO_DIR / "datasets/processed/emotional_stories.jsonl"
print(f"Local output: {output_path}")
print(f"Exists: {output_path.exists()}")

if output_path.exists():
    !wc -l {output_path}
    !head -n 2 {output_path}

if DRIVE_OUTPUT_DIR is not None and output_path.exists():
    target_path = DRIVE_OUTPUT_DIR / output_path.name
    shutil.copy2(output_path, target_path)
    print(f"Copied to: {target_path}")